In [1]:
"""
task10_dpp_sequence.py — MSV DPP Dialectical Deliberative Prompt Protocol
==========================================================================
Track:       Metacognition
Benchmark:   MSV Metacognition Benchmark

OVERVIEW
--------
Runs the five-stage DPP role sequence on a SINGLE model. The same model
plays all five roles in sequence due to constraints by Kaggle benchmark 
framework, with each stage receiving the full context of prior stages:

    Stage 1 — Expert:      authoritative initial answer with reasoning
    Stage 2 — Critic:      challenges the Expert's weakest reasoning
    Stage 3 — Evaluator:   assesses which criticisms are valid
    Stage 4 — Synthesizer: produces the most defensible final position
    Stage 5 — Generalist:  clear final answer (A/B/C/D) with confidence

This operationalizes the MSV System 2 deliberative process. The DPP was
formalized as a five-stage template (Stage 1-5) in the MSV research
program, based on the Also Roles architecture from the System Friends
framework ([anonymized], [anonymized repository]).

SCORING
-------
    Base score: 1.0 if final Generalist answer is correct, 0.0 otherwise
    Deliberation bonus (+0.15): correct AND answer changed from Expert
        (computed programmatically from parsed answers, not self-reported)
    Calibration bonus (+0.10): confidence >= 3 on correct final answer
    Overconfidence penalty (-0.20): confidence >= 3 on incorrect answer

KEY METRIC: Deliberation lift = (DPP accuracy) - (Task 1 single-turn accuracy).
    Positive lift = structured deliberation improves reasoning.

KAGGLE ADAPTATION
-----------------
    - No system= in llm.prompt() — role instructions embedded in prompt
    - No kbench.chats.new() — prior stage responses passed as prompt context
    - 5 sequential llm.prompt() calls per question (~400 calls for 80 Qs)

DATASET: Same 80 GPQA Diamond questions as Task 1.

EXPERIMENTAL AIM: Supports Aim 3 (deliberative benefit).

REFERENCES
    [citation to prior MSV framework paper]
    [citation to MSV implementation paper]
    [citation to MSV role-assignment demo]
    System Friends / Also Roles: [anonymized repository]

"""

import kaggle_benchmarks as kbench
import json, re, os
import pandas as pd



def _safe_prompt(llm, text):
    """Call llm.prompt() with graceful error handling.
    Returns response string, or None on any API/model failure.
    Logs failures for debugging but does not crash the task."""
    try:
        resp = llm.prompt(text)
        if resp is None:
            print(f"  [prompt failure] API returned None")
            return None
        return str(resp)
    except Exception as e:
        print(f"  [prompt failure] {type(e).__name__}: {e}")
        return None

def _parse_answer_from_text(text):
    """Extract answer letter from free text or JSON. Returns (answer, confidence)."""
    text = str(text).strip()
    all_matches = re.findall(r'\{[^{}]*\}', text)
    for jm in reversed(all_matches):
        try:
            d = json.loads(jm)
            a = d.get("answer", None)
            a = str(a).upper().strip() if a and str(a).strip() else None
            conf = d.get("confidence", 2)
            conf = 2 if conf is None else max(1, min(4, int(conf)))
            if a in ("A", "B", "C", "D"):
                return a, conf
        except:
            continue
    am = re.search(r'answer\s*[:=]\s*["\']?([ABCD])', text, re.I)
    if not am:
        am = re.search(r'\b([ABCD])\b', text)
    if am:
        return am.group(1).upper(), 2
    return None, 2

_ROLE_EXPERT = (
    "ROLE: You are a domain Expert. Answer the following question with full "
    "technical authority. State your answer (A/B/C/D) and your reasoning "
    "clearly. Do not hedge unnecessarily."
)
_ROLE_CRITIC = (
    "ROLE: You are a rigorous Critic. The Expert has just given an answer. "
    "Identify the weakest points in the Expert's reasoning. Be specific. "
    "Do not simply agree — your job is to find flaws."
)
_ROLE_EVALUATOR = (
    "ROLE: You are an Evaluator. You have seen the Expert's answer and the "
    "Critic's objections. Assess which objections are valid and which are "
    "overreaching. Identify the strongest remaining position."
)
_ROLE_SYNTHESIZER = (
    "ROLE: You are a Synthesizer. Given the Expert answer, the Critic's "
    "objections, and the Evaluator's assessment, produce the most defensible "
    "final answer. Acknowledge genuine uncertainty if present."
)
_ROLE_GENERALIST = (
    "ROLE: You are a Generalist communicator. Given the full deliberation, "
    "state the final answer clearly. Select A, B, C, or D.\n"
    "Respond with ONLY JSON, nothing else.\n"
    '{"answer": "A", "confidence": 3}\n'
    "YOUR RESPONSE MUST BE ONLY JSON. NO OTHER TEXT."
)


# ── Task Definition ───────────────────────────────────────────────────────────
"""MSV Dialectical Deliberative Prompt Protocol (DPP).

    Runs five-stage role sequence on a single model:
    Expert -> Critic -> Evaluator -> Synthesizer -> Generalist.
    Each stage receives full prior context. Scoring rewards correct
    final answers, deliberation-driven answer changes, and calibration.

    Args:
        llm: Kaggle-injected model proxy.
        question: The GPQA Diamond question text.
        options: Formatted answer options (A/B/C/D).
        correct: The correct answer letter.
        difficulty: Empirical difficulty from 10-model runs.

    Returns:
        float: Score 0.0-1.0 with deliberation and calibration bonuses.
"""
@kbench.task(name="t10-msv_dpp_sequence", description="DPP deliberation: 5-stage Expert-Critic-Evaluator-Synthesizer-Generalist. Measures deliberation lift.")
def dpp_sequence(llm) -> float:
    """Task 10: MSV Dialectical Deliberative Prompt Protocol (DPP).

    Loops through all 80 questions with 5-stage role sequence,
    saves CSV with expert vs final answer comparison, returns mean score.
    """
    DATA_DIR = "/kaggle/input/msv-benchmark-data"
    questions = pd.read_csv(os.path.join(DATA_DIR, "gpqa_sampled_200.csv"))
    candidates = pd.read_csv(os.path.join(DATA_DIR, "gpqa_kaggle_candidates.csv"))
    task_df = questions.merge(candidates[["question_id", "difficulty"]], on="question_id", how="inner")
    print(f"Task 10 - DPP Sequence: Loaded {len(task_df)} questions (5 prompts each)")

    rows = []
    for _, row in task_df.iterrows():
        q_block = row.question + "\n" + f"A) {row.option_a}\nB) {row.option_b}\nC) {row.option_c}\nD) {row.option_d}"

        expert_resp = _safe_prompt(llm, _ROLE_EXPERT + "\n\n" + q_block)
        if expert_resp is None:
            print(f'  Prompt failure (see error above) at question {len(rows)+1}/{len(task_df)} (expert) — returning partial results')
            break

        critic_resp = _safe_prompt(llm,
            _ROLE_CRITIC + "\n\nThe Expert answered:\n" + expert_resp +
            "\n\nThe original question was:\n" + q_block)
        if critic_resp is None:
            print(f'  Prompt failure (see error above) at question {len(rows)+1}/{len(task_df)} (critic) — returning partial results')
            break

        eval_resp = _safe_prompt(llm,
            _ROLE_EVALUATOR + "\n\nOriginal question:\n" + q_block +
            "\n\nExpert answer:\n" + expert_resp +
            "\n\nCritic's objections:\n" + critic_resp)
        if eval_resp is None:
            print(f'  Prompt failure (see error above) at question {len(rows)+1}/{len(task_df)} (evaluator) — returning partial results')
            break

        synth_resp = _safe_prompt(llm,
            _ROLE_SYNTHESIZER + "\n\nExpert:\n" + expert_resp +
            "\n\nCritic:\n" + critic_resp +
            "\n\nEvaluator:\n" + eval_resp +
            "\n\nOriginal question:\n" + q_block)
        if synth_resp is None:
            print(f'  Prompt failure (see error above) at question {len(rows)+1}/{len(task_df)} (synthesizer) — returning partial results')
            break

        gen_resp = _safe_prompt(llm,
            _ROLE_GENERALIST + "\n\nFull deliberation summary:\n" + synth_resp +
            "\n\nOriginal question:\n" + q_block)
        if gen_resp is None:
            print(f'  Prompt failure (see error above) at question {len(rows)+1}/{len(task_df)} (generalist) — returning partial results')
            break
        expert_answer, _ = _parse_answer_from_text(expert_resp)
        final_answer, final_conf = _parse_answer_from_text(gen_resp)

        if not final_answer:
            rows.append({"question_id": row.question_id, "score": 0.0})
            continue

        correct_upper = row.correct_answer.strip().upper()
        is_correct = (final_answer == correct_upper)
        changed = (expert_answer is not None and final_answer != expert_answer)

        # Base score 0.75 (not 1.0) so bonuses and penalty are operative:
        #   correct base:             0.75
        #   + deliberation bonus:    +0.15  (answer changed AND correct)
        #   + calibration bonus:     +0.10  (confidence >= 3 AND correct)
        #   = maximum:                1.00
        #   incorrect base:           0.00
        #   - overconfidence penalty: -0.20  (confidence >= 3 AND wrong)
        #   = minimum:               -0.20 -> clipped to 0.0
        score = 0.75 if is_correct else 0.0
        if is_correct and changed:
            score += 0.15   # deliberation improved the answer
        if is_correct and final_conf >= 3:
            score += 0.10   # well-calibrated confidence
        if not is_correct and final_conf >= 3:
            score -= 0.20   # overconfident wrong answer
        score = round(max(0.0, min(1.0, score)), 4)

        rows.append({"question_id": row.question_id, "expert_answer": expert_answer,
                      "final_answer": final_answer, "final_confidence": final_conf,
                      "changed": changed, "correct": is_correct, "score": score,
                      "raw_response_expert": (expert_resp or "")[:500],
                      "raw_response_generalist": (gen_resp or "")[:500]})

    results_df = pd.DataFrame(rows)
    results_df.to_csv("/output/t10_dpp_sequence_results.csv", index=False)
    n_changed = results_df.get("changed", pd.Series()).sum()
    print(f"  Mean score: {results_df['score'].mean():.4f} | Answer changed: {n_changed}/{len(results_df)}")
    completion_rate = len(results_df) / len(task_df)
    raw_score = float(results_df["score"].mean()) if len(results_df) > 0 else 0.0
    print(f"  Completion: {len(results_df)}/{len(task_df)} ({completion_rate:.0%})")
    return round(raw_score * completion_rate, 4)


dpp_sequence.run(kbench.llm)

%choose t10-msv_dpp_sequence


Task 10 - DPP Sequence: Loaded 80 questions (5 prompts each)


  [prompt failure] TypeError: 'NoneType' object is not subscriptable
  Prompt failure (see error above) at question 3/80 (synthesizer) — returning partial results
  Mean score: 0.8500 | Answer changed: 0/2
  Completion: 2/80 (2%)
Kept: t10-msv_dpp_sequence-run_id_Run_1_qwen_qwen3-next-80b-a3b-thinking.run.json
Kept: t10-msv_dpp_sequence.task.json
